第09回講義
========

Seaborn
-------

[Seaborn](https://seaborn.pydata.org)とは、matplotlibのプロット機能を拡張したモジュールで、pandasなどと組み合わせてデータの可視化を容易にします。以下では、これまでの復習も兼ねながら、seabornでプロットやデータ可視化、データサイエンスを実践してみます。

seabornには以下のようなプロットメソッドがあります。(使い方はネットで検索、あるいはドキュメンテーションで確認します。)

![seaborn_plots](https://seaborn.pydata.org/_images/function_overview_8_0.png)


<mark>練習1</mark> `tokyo-temp.csv`のCSVファイルからpandasのデータフレームを作成し、seabornで折れ線グラフと散布図でプロットしなさい。

In [ ]:
import pandas as pd
import seaborn as sns

df = pd.read_csv('tokyo-temp.csv')

sns.lineplot(data=df, x='year', y='temp')
#sns.scatterplot(data=df, x='year', y='temp')
#sns.relplot(data=df, x='year', y='temp', kind='line')

<mark>練習2</mark> pandasの機能を使って`tokyo-temp.csv`の移動平均を作成し、seabornを使って重ねてプロットしなさい。

In [ ]:
import pandas as pd
import seaborn as sns

# 1列目のtempをインデックスにすると自動的にx軸の値になる
df = pd.read_csv('tokyo-temp.csv', index_col=0)

width = 5 # 移動平均を計算するウィンドウ幅
df['sma'] = df['temp'].rolling(width).mean()

sns.lineplot(data=df)

<mark>練習3</mark> 移動平均線5年、10年、15年度重ねてプロットしなさい。

In [ ]:
import pandas as pd
import seaborn as sns

# 1列目のyearをインデックスにすると自動的にx軸の値になる
df = pd.read_csv('tokyo-temp.csv', index_col=0)

width = [5, 10, 15] # 移動平均を計算するウィンドウ幅

for w in width:
    df[f'sma ({w} years)'] = df['temp'].rolling(w).mean()

sns.lineplot(data=df)

seabornで回帰分析(regression)を含めてプロットするには、regplotメソッドを使用します。

<mark>練習4</mark> seabornのregplotを用いて、`tokyo-temp.csv`データに対する回帰分析を行いなさい。

In [ ]:
import pandas as pd
import seaborn as sns

# xとyを列として指定したいので、yearをインデックスにしない
df = pd.read_csv('tokyo-temp.csv')

sns.regplot(data=df, x='year', y='temp', marker='x', line_kws={'color': 'red'})

より高次の多項式による回帰分析では、`order`オプションで次数を指定します。

In [ ]:
import pandas as pd
import seaborn as sns

df = pd.read_csv('tokyo-temp.csv')

sns.regplot(data=df, x='year', y='temp', order=3)

残念ながら`regplot`には回帰分析の結果(係数)を出力する機能はありませんが、プロットオブジェクトから回帰曲線部分のみを取り出すことはできます。

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('tokyo-temp.csv')

p = sns.regplot(data=df, x='year', y='temp', order=3)
plt.show()

plt.plot(p.get_lines()[0].get_xdata(), p.get_lines()[0].get_ydata(), color='red')

ペアプロットとクラス分類
------------------

### Irisデータ

統計学に多大な功績を残したロナルド・エイルマー・フィッシャー(Ronald Aylmer Fisher)が論文で使用したデータセットであることから、フィッシャーのIrisデータセット(Fisher’s Iris Dataset)と呼ばれることもあります。

Irisデータセットでは、アヤメ科アヤメ属の中でsetosa、versicolor、virginicaという3種の花に着目しています。

setosa、versicolor、virginicaは、素人にはほとんど同じような花に見えます。しかしがく片（sepal）の長さ、幅、および花弁（petal）の長さ、幅を測定、分析することによって、それらを区別するための特徴が見えてきます。setosa、versicolor、virginicaそれぞれ、50個体について測定しています。

![Iris](iris.png)


`iris.csv`がそのデータセットです。一部を抜き出していますが、全部で150行5列あります。

<mark>練習5</mark> `iris.csv`データをpandasのデータフレームとして読み込み、その中身を確認しなささい。

In [ ]:
import pandas as pd

iris = pd.read_csv("iris.csv")

iris.head()

<mark>練習6</mark> `iris.csv`データをpandasのデータフレームとして読み込み、各種統計データおよび相関行列を表示させなさい。

In [ ]:
iris.describe()

In [ ]:
iris[['sepal_length','sepal_width','petal_length','petal_width']].corr()

<mark>練習7</mark> seabornの`pairplot`関数を用いると、ペアプロット(データの組み合わせごとの散布図)を表示させなさい。

In [ ]:
import pandas as pd
import seaborn as sns

iris = pd.read_csv("iris.csv")

sns.pairplot(iris, hue='species')

他にも`jointplot`という機能があります。

In [ ]:
sns.jointplot(iris, kind='scatter', x='petal_length', y='petal_width', hue='species')

データの分布については、カーネル密度推定(KDE: Kernel Density Estimation)と呼ばれる手法を用いて、等高線表示もできる。

In [ ]:
sns.jointplot(iris, kind='kde', x='petal_length', y='petal_width', hue='species')

### Titanicデータ

タイタニック号の沈没は、歴史上最も有名な海難事故の 1 つです。

初航海中の 1912年4月15日、「不沈」と広く考えられていた RMS タイタニック号が氷山に衝突して沈没しました。残念ながら、乗船していた全員に十分な数の救命ボートがなかったため、2224人の乗客と乗組員のうち1502人が死亡しました。

生存には運の要素もありましたが、一部の人々のグループは他のグループよりも生き残る可能性が高かったようです。

この課題では、乗客データ (名前、年齢、性別、社会経済階級など) を使用して、「どのような人が生き残る可能性が高いか?」という質問に答える予測モデルを構築します。

<mark>練習8</mark> 'titanic.csv'のデータをpandasのデータフレームとして読み込み、その中身を確認しなさい。

In [ ]:
import pandas as pd

titanic = pd.read_csv('titanic.csv')

titanic.head()

<mark>練習9</mark> `titanic.csv`のデータをpandasのデータフレームとして読み込み、各種統計データを表示させなさい。

In [ ]:
titanic.describe()

### カテゴリーごとの集計と棒グラフ

<mark>練習10</mark> 乗客の旅客クラスごとの人数を集計して棒グラフとして表しなさい。

In [ ]:
import pandas as pd
import seaborn as sns

titanic = pd.read_csv('titanic.csv')

sns.catplot(x='pclass', data=titanic, kind='count')

<mark>練習11</mark> seabornの`histplot`機能を用いて、年齢(age)分布について、ヒストグラムをKDEプロットも合わせて表示させなさい。

In [ ]:
sns.histplot(titanic['age'], kde=True)

<mark>練習12</mark> seabornの`catplot`を用いて、箱ひげ図を表示させなさい。

In [ ]:
sns.catplot(x='gender', y='age', data=titanic, kind='box', hue='survived')

<mark>練習13</mark> ヒストグラムを性別(gender)ごとに重ねて表示させ、年齢構成に違いがあったのか調べなさい。

In [ ]:
sns.histplot(data=titanic, x='age', hue='gender')

他にも、箱ひげ図にKDE分布も含めたプロット(バイオリンプロット)等も可能。

In [ ]:
sns.catplot(x='gender', y='age', data=titanic, kind='violin', hue='survived', split=True)